In [1]:
%load_ext autoreload
%autoreload 2

import os
print(os.getcwd())
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn, optim
from torch.utils.data import DataLoader
from torch.optim import Adam,AdamW
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import pickle

from utils import simdatset, reproducibility, calculate_evaluation_metrics

batch_size = 256
reproducibility(2025)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/disk1/user/liaoshuilin/project/35.TAPE_EXO/assay_diffusion


# Load data

In [ ]:
with open(f'../result/data_realEV/Real_data.pkl', 'rb') as file:
    loaded_data = pickle.load(file)
all_genename = loaded_data['all_genename']
celltypes = loaded_data['celltypes']
real_x = loaded_data['real_x']
real_x_healthy = loaded_data['real_x_healthy']
real_x_benign = loaded_data['real_x_benign']
real_x_brca = loaded_data['real_x_brca']
real_x_hcc = loaded_data['real_x_hcc']
real_x_paad = loaded_data['real_x_paad']

# Get sigmatrix

In [ ]:
# %load_ext autoreload
# %autoreload 2
# from train_re import evaluation
# with open(f'../result/data_abalation/Stim_data.pkl', 'rb') as file:
#     loaded_data = pickle.load(file)
# GTE_x_train = loaded_data['GTE_x_train']
# GTE_y_train = loaded_data['GTE_y_train']
# model = torch.load("/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_abalation/model_stage1.pth", weights_only=False, map_location=device)
# train_loader2 = DataLoader(simdatset(GTE_x_train, GTE_y_train), batch_size=batch_size, shuffle=False)
# x_recon_tr, f_tr, z_tr  = evaluation(train_loader2, model, device=device)
# sigmatrix = np.linalg.pinv(f_tr) @ x_recon_tr 

sigmatrix =  pd.read_csv('/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_abalation/sigmatrix.csv')

# All real EV

In [ ]:
%load_ext autoreload
%autoreload 2
from train_re import adaptive_stage_domain9
from utils import calculate_evaluation_metrics
reproducibility(2025)

out_pth = "/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_realEV/real_"
x_recon_HPA, f_HPA, z_HPA, model2 = adaptive_stage_domain9(x=real_x, model_name="/disk1/user/liaoshuilin/project/35.TAPE_EXO/result/model_abalation/model_stage1", mode = 'overall5', steps=10, max_iter=40, device=device, sigmatrix = sigmatrix)
# pd.DataFrame(x_recon_HPA).to_csv(out_pth + 'x_recon.csv', index=False, header=False)
pd.DataFrame(f_HPA).to_csv(out_pth + 'y.csv', index=False, header=False)
rmse_m, average_rmse, pearson_corr_value, average_pearson_corr, mae_m, average_mae, ccc_values, average_ccc = calculate_evaluation_metrics(x_recon_res = x_recon_HPA, input_X = real_x, out_pth = out_pth)
print(f"RMSE of x: {average_rmse:.4f}\nPCC of x: {average_pearson_corr:.4f}\nMAE of x: {average_mae:.4f}\nCCC of x: {average_ccc:.4f}")
torch.save(model2, out_pth + "model_stage2.pth")